# Day 03：logits 与 softmax

对应 [`docs/day03-04.md`](../docs/day03-04.md) 的**任务 2**（四个实验）和**任务 5**（weight tying）。
笔记见 [`docs/concepts/02-lm-head.md`](../docs/concepts/02-lm-head.md)。

`LMHead` 要到任务 3 才实现，所以前四节用 `torch.randn` 造假 logits——
softmax 的性质与 `LMHead` 写得对不对无关，用随机数反而更干净。

**本节要验证的核心结论**：softmax 是单调变换，不改变排序，
所以贪心解码根本不需要算 softmax。

## 1. logits 不是概率

打印一行 logits 的 min / max / sum，确认三件事：可以是负数、可以大于 1、总和不等于 1。

In [ ]:
import torch

# 造一行假 logits（V=1000），打印 min / max / sum
t = torch.randn(20)
print("=========> init", t)
print("=========> min", t.min())
print("=========> max", t.max())
print("=========> sum", t.sum())

## 2. softmax 之后才是概率

对同一行做 softmax，确认全部落在 `[0, 1]` 且和为 1。

用 `torch.allclose` 判断和是否为 1，不要用 `==`——浮点求和有累积误差。

In [ ]:
# softmax 后检查取值范围和求和
import torch

t = torch.randn(20)

p = torch.softmax(t, dim=-1)
print(torch.allclose(p.sum(), torch.tensor(1.0)))

## 3. argmax 不变（本节重点）

验证 `logits.argmax(-1)` 和 `softmax(logits).argmax(-1)` 完全相同。

用多个 batch、多个位置一起验，而不是只看一个数——单个样本相等可能是碰巧。

In [ ]:
# 在 [B, S, V] 上比较两种 argmax 是否逐元素相同
import torch

t = torch.randn(20)

p = torch.softmax(t, dim=-1)
# print(torch.allclose(p.sum(), torch.tensor(1.0)))
print(t.argmax(-1))
print(torch.softmax(t, dim=-1).argmax(-1))

## 4. 温度

对 `logits / T` 做 softmax，比较 `T=0.5 / 1.0 / 2.0` 的分布尖锐程度。

用一个量化指标描述「尖锐」，比如最大概率值、或前 5 个概率之和。

顺便验证一件事：**除以 T 会改变分布形状，但不改变 argmax**（`T > 0` 时它仍是单调的）。
对比第 3 节——加一个常数和乘一个常数，对分布的影响并不一样。

In [ ]:
import torch

torch.manual_seed(0)
logits = torch.randn(20)


def sharpness(p):
    """三个角度描述「尖锐」：最大概率、前 5 之和、熵（熵越小越尖）。"""
    top5 = p.topk(5).values.sum()
    entropy = -(p * p.clamp_min(1e-12).log()).sum()
    return p.max().item(), top5.item(), entropy.item()


print(f"{'T':>5} | {'max p':>7} | {'top5 和':>7} | {'熵':>6} | argmax")
print("-" * 46)
for T in (0.5, 1.0, 2.0, 100.0):
    p = torch.softmax(logits / T, dim=-1)
    max_p, top5, entropy = sharpness(p)
    print(f"{T:>5} | {max_p:>7.4f} | {top5:>7.4f} | {entropy:>6.4f} | {p.argmax(-1).item()}")

print(f"\n均匀分布的熵上界 log(20) = {torch.tensor(20.0).log():.4f}")
print(f"原始 argmax = {logits.argmax(-1).item()}，四个 T 全部一致 —— 温度不改变排序")

# 加常数 vs 乘常数：前者连分布都不变，后者只保排序
base = torch.softmax(logits, dim=-1)
added = torch.softmax(logits + 100, dim=-1)
scaled = torch.softmax(logits * 100, dim=-1)

print("\n=== 加常数 vs 乘常数 ===")
print(f"softmax(x + 100) == softmax(x) : {torch.allclose(base, added)}")
print(f"softmax(x * 100) == softmax(x) : {torch.allclose(base, scaled)}")
print(f"  乘 100 后 max p = {scaled.max():.6f}，分布塌成了 one-hot")
print(f"argmax 三者仍然一致: {base.argmax(-1).item()} / {added.argmax(-1).item()} / {scaled.argmax(-1).item()}")

## 5. Weight tying 与参数量（任务 5）

三件事：

1. 用 `is` 确认两个 `weight` 是同一个对象，不只是数值相等
2. 改动其中一个，另一个跟着变
3. 统计参数量。`nn.Module.parameters()` 会自动去重共享张量，所以
   `sum(p.numel() for p in model.parameters())` 在 `tie_weights()` 前后应该直接减半

GPT-2 尺度参考：untied 77.2M（fp32 294 MB），tied 38.6M（fp32 147 MB）。

In [ ]:
# tie_weights() 前后：is 判断、联动验证、参数量对比
